In [1]:
import os
import sys
import json
import time
import signal
import subprocess
from pathlib import Path

REPO = "https://github.com/eslam-ahmed43/llm-lab.git"
ROOT = Path("/kaggle/working/llm-lab")
if not ROOT.exists():
    subprocess.run(["git", "clone", REPO, str(ROOT)], check=True)
else:
    subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
os.chdir(ROOT)
(ROOT / "logs").mkdir(exist_ok=True)
assert (ROOT / "src/exp6_profiling.py").exists(), "exp6_profiling.py is missing: push it to GitHub first"

os.environ["HF_HOME"] = "/tmp/hf_cache"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                     shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("git log --oneline -1; ls src", shell=True, capture_output=True, text=True).stdout)
print(sys.version)

Cloning into '/kaggle/working/llm-lab'...


Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

2c8809d Exp 6: add memory bandwidth floor
bench_inference.py
exp5_internals.py
exp6_profiling.py
hf_batched_server.py
hf_server.py
plot_lora.py
quantize.py
train_lora.py

3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


In [2]:
%%bash
set -e
python -c "
import transformers, torch
print('transformers', transformers.__version__, '| torch', torch.__version__)
from huggingface_hub import snapshot_download
snapshot_download('Qwen/Qwen2.5-1.5B-Instruct')
print('model cached')
"

transformers 5.16.1 | torch 2.11.0+cu128
model cached


Fetching 10 files: 100%|██████████| 10/10 [00:07<00:00,  1.31it/s]


In [3]:
def run_cmd(tag, args, timeout_s=1800):
    log_path = ROOT / "logs" / f"exp6_{tag}.log"
    cmd = [sys.executable, "-X", "faulthandler", "-u", str(ROOT / "src/exp6_profiling.py")] + args
    env = {**os.environ, "HF_HUB_DISABLE_PROGRESS_BARS": "1", "PYTHONUNBUFFERED": "1"}

    def new_lines(shown):
        lines = log_path.read_text(errors="replace").splitlines()
        for line in lines[shown:]:
            if line.strip() and "Loading weights" not in line and "unauthenticated" not in line:
                print(line[:600], flush=True)
        return len(lines)

    start, shown = time.time(), 0
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env, cwd=str(ROOT))
        while proc.poll() is None:
            time.sleep(5)
            shown = new_lines(shown)
            if time.time() - start > timeout_s:
                os.kill(proc.pid, signal.SIGABRT)
                time.sleep(3)
                proc.kill()
                print(log_path.read_text(errors="replace")[-3000:])
                raise TimeoutError(f"{tag} exceeded {timeout_s}s")
    new_lines(shown)
    if proc.returncode != 0:
        print(log_path.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"{tag} failed with exit code {proc.returncode}")

In [4]:
run_cmd("selftest", ["selftest"])

{'dynamic_nosync': True, 'static_eager': True, 'first_step_logit_diff': 5.364418029785156e-07}
SELFTEST PASSED


In [5]:
run_cmd("full", ["run", "--check", "--bench", "--profile", "--repeats", "5",
                 "--out", str(ROOT / "results/profiling_exp6.json")], timeout_s=2400)

[check] fp32 equivalence of all variants
   {'dynamic_nosync': True, 'static_eager': True, 'static_graph': True}
[fp16] building the model
  weights 3.09 GB, measured copy bandwidth 243 GB/s, so streaming the weights once takes at least 12.7 ms
  dynamic_eager_sync: 27.12 ms/step (enqueue alone 27.12 ms), 36.9 tokens/s
  dynamic_eager_nosync: 24.52 ms/step (enqueue alone 24.50 ms), 40.8 tokens/s
  static_eager_nosync: 25.08 ms/step (enqueue alone 25.07 ms), 39.9 tokens/s
  static_cuda_graph: 21.22 ms/step (enqueue alone 20.26 ms), 47.1 tokens/s
/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
  profile dynamic_eager_nosync: {'device_events_per_step': 1312.0333333333333, 'device_busy_ms_per_step': 17.51790653333314, 'top_device_ops_ms_per_step': [['void gemv2T_kernel_val<int, int,

In [6]:
import zipfile

with zipfile.ZipFile("/kaggle/working/llm-lab-exp6.zip", "w", zipfile.ZIP_DEFLATED) as z:
    files = sorted((ROOT / "results").glob("profiling_*.json")) + sorted((ROOT / "logs").glob("exp6_*.log"))
    for f in files:
        z.write(f, arcname=f"{f.parent.name}/{f.name}")
print("Zipped:", zipfile.ZipFile("/kaggle/working/llm-lab-exp6.zip").namelist())

Zipped: ['results/profiling_exp6.json', 'logs/exp6_full.log', 'logs/exp6_selftest.log']
